# 01 - Data Preprocessing

Cleans the three raw DineSmart datasets and writes analysis-ready tables to `Data/processed/`:

1. **Zomato_Database** (`users`, `restaurant`, `food`, `menu`, `orders`) - the relational core used for customer segmentation and restaurant/food performance analysis.
2. **Food Delivery Order History Data** - order-level data with itemized baskets and ratings, used for market basket analysis and order/revenue analysis.
3. **Zomato Delivery Operations Analytics Dataset** - delivery-leg data (weather, traffic, delivery time), used for delivery performance analysis.

Each section loads the raw CSV, fixes types, resolves missing/malformed values, and enforces referential integrity where relevant. The final section derives customer-level RFM features for the K-Means segmentation step and saves every cleaned table.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)

RAW = Path("../Data/raw")
PROCESSED = Path("../Data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

## 1. Zomato_Database - Users

- Drop the `password` column (credential-like data with no analytical use).
- Normalize column names to snake_case.
- No missing values in this table; `user_id` is a clean primary key (no duplicates).

In [2]:
users = pd.read_csv(RAW / "Zomato_Database/users.csv")

users = users.drop(columns=["Unnamed: 0", "password"])
users.columns = users.columns.str.strip().str.lower().str.replace(" ", "_")
users = users.drop_duplicates(subset="user_id").reset_index(drop=True)

for col in ["gender", "marital_status", "occupation", "monthly_income", "educational_qualifications"]:
    users[col] = users[col].astype("category")

print(users.shape)
print(users.isna().sum())
users.head()

(100000, 10)
user_id                       0
name                          0
email                         0
age                           0
gender                        0
marital_status                0
occupation                    0
monthly_income                0
educational_qualifications    0
family_size                   0
dtype: int64


,user_id,name,email,age,gender,marital_status,occupation,monthly_income,educational_qualifications,family_size
0,1,Claire Ferguson,fordanthony@example.net,20,Female,Single,Student,No Income,Post Graduate,4
1,2,Jennifer Young,ann96@example.com,24,Female,Single,Student,Below Rs.10000,Graduate,3
2,3,Jermaine Roberson,uwalker@example.org,22,Male,Single,Student,Below Rs.10000,Post Graduate,3
3,4,Rachel Carpenter,kimberlypatterson@example.net,22,Female,Single,Student,No Income,Graduate,6
4,5,Shawn Parker,daniellebennett@example.com,22,Male,Single,Student,Below Rs.10000,Post Graduate,4


## 2. Zomato_Database - Restaurants

- Drop `lic_no`, `link`, `menu` (operational/legal metadata not used in analysis).
- Drop the 86 rows with no `name` (unusable records).
- `rating`: `'--'` means "not yet rated" -> convert to `NaN`.
- `rating_count`: parse free text (`"50+ ratings"`, `"1K+ ratings"`, `"Too Few Ratings"`) into an approximate numeric lower bound.
- `cost`: strip the `₹` sign and convert to a numeric `cost_for_two`.
- Missing `cuisine` -> `"Unknown"`.

In [3]:
restaurant = pd.read_csv(RAW / "Zomato_Database/restaurant.csv")

restaurant = restaurant.drop(columns=["Unnamed: 0", "lic_no", "link", "menu"])
restaurant.columns = restaurant.columns.str.strip().str.lower()
restaurant = restaurant.dropna(subset=["name"]).copy()

restaurant["rating"] = pd.to_numeric(restaurant["rating"].replace("--", np.nan))


def parse_rating_count(val):
    if pd.isna(val) or val == "Too Few Ratings":
        return 0
    val = val.replace("ratings", "").replace("+", "").strip()
    if val.endswith("K"):
        return int(float(val[:-1]) * 1000)
    return int(val)


restaurant["rating_count"] = restaurant["rating_count"].apply(parse_rating_count)

restaurant["cost_for_two"] = (
    restaurant["cost"].str.replace("₹", "", regex=False).str.strip().astype(float)
)
restaurant = restaurant.drop(columns=["cost"])

restaurant["cuisine"] = restaurant["cuisine"].fillna("Unknown")
restaurant = restaurant.rename(columns={"id": "r_id"})
restaurant = restaurant.drop_duplicates(subset="r_id").reset_index(drop=True)

print(restaurant.shape)
print(restaurant.isna().sum())
restaurant.head()

(148455, 8)
r_id                0
name                0
city                0
rating          87014
rating_count        0
cuisine             0
address             0
cost_for_two       45
dtype: int64


,r_id,name,city,rating,rating_count,cuisine,address,cost_for_two
0,567335,AB FOODS POINT,Abohar,NaN,0,"Beverages,Pizzas","AB FOODS POINT, NEAR RISHI NARANG DENTAL CLINI...",200.0
1,531342,Janta Sweet House,Abohar,4.4,50,"Sweets,Bakery","Janta Sweet House, Bazar No.9, Circullar Road,...",200.0
2,158203,theka coffee desi,Abohar,3.8,100,Beverages,"theka coffee desi, sahtiya sadan road city",100.0
3,187912,Singh Hut,Abohar,3.7,20,"Fast Food,Indian","Singh Hut, CIRCULAR ROAD NEAR NEHRU PARK ABOHAR",250.0
4,543530,GRILL MASTERS,Abohar,NaN,0,"Italian-American,Fast Food","GRILL MASTERS, ADA Heights, Abohar - Hanumanga...",250.0


## 3. Zomato_Database - Food

Only 1 row has a missing `item`/`veg_or_non_veg` - drop it. `f_id` is already a clean, duplicate-free key.

In [4]:
food = pd.read_csv(RAW / "Zomato_Database/food.csv")

food = food.drop(columns=["Unnamed: 0"])
food = food.dropna(subset=["item", "veg_or_non_veg"]).copy()
food = food.drop_duplicates(subset="f_id").reset_index(drop=True)

print(food.shape)
print(food.isna().sum())
food.head()

(371560, 3)
f_id              0
item              0
veg_or_non_veg    0
dtype: int64


,f_id,item,veg_or_non_veg
0,fd0,Aloo Tikki Burger,Veg
1,fd1,Veg Creamy Burger,Veg
2,fd2,Cheese Burst Burger,Veg
3,fd3,Paneer Creamy Burger,Veg
4,fd4,Maxican Burger,Veg


## 4. Zomato_Database - Menu

- `price` is stored as text and 1,193 rows use a `"₹200 FOR TWO"` format instead of a plain number - extract the numeric value with a regex for every row.
- Drop the ~0.02% of rows whose `r_id`/`f_id` don't exist in the cleaned `restaurant`/`food` tables (broken foreign keys).

In [5]:
menu = pd.read_csv(RAW / "Zomato_Database/menu.csv", low_memory=False)

menu = menu.drop(columns=["Unnamed: 0"])


def parse_price(val):
    match = re.search(r"[\d.]+", str(val))
    return float(match.group()) if match else np.nan


menu["price"] = menu["price"].apply(parse_price)

menu = menu[
    menu["r_id"].isin(restaurant["r_id"]) & menu["f_id"].isin(food["f_id"])
].copy()
menu = menu.drop_duplicates().reset_index(drop=True)

print(menu.shape)
print(menu.isna().sum())
menu.head()

(1178743, 5)
menu_id    0
r_id       0
f_id       0
cuisine    0
price      0
dtype: int64


,menu_id,r_id,f_id,cuisine,price
0,mn0,567335,fd0,"Beverages,Pizzas",40.0
1,mn0,567335,fd669322,"Beverages,Pizzas",40.0
2,mn328,158203,fd0,Beverages,65.0
3,mn328,158203,fd669322,Beverages,65.0
4,mn449,158203,fd0,Beverages,65.0


## 5. Zomato_Database - Orders

- Parse `order_date` to a real datetime.
- Drop the 1,617 rows (~1.1%) with a missing `r_id` - revenue/order records can't be attributed to a restaurant without it.
- Drop the 1,611 rows with `sales_amount <= 0` - these are refunds/cancellations, not completed sales.
- Enforce referential integrity against the cleaned `users` and `restaurant` tables.

In [6]:
orders = pd.read_csv(RAW / "Zomato_Database/orders.csv")

orders = orders.drop(columns=["Unnamed: 0"])
orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")

orders = orders.dropna(subset=["r_id"]).copy()
orders["r_id"] = orders["r_id"].astype(int)

orders = orders[orders["sales_amount"] > 0].copy()
orders = orders[
    orders["user_id"].isin(users["user_id"]) & orders["r_id"].isin(restaurant["r_id"])
].copy()

print(orders.shape)
print(orders.isna().sum())
orders.head()

(146979, 6)
order_date      0
sales_qty       0
sales_amount    0
currency        0
user_id         0
r_id            0
dtype: int64


,order_date,sales_qty,sales_amount,currency,user_id,r_id
0,2017-10-10,100,41241,INR,49226,567335
2,2018-04-06,1,875,INR,5321,158203
3,2018-04-11,1,583,INR,21343,187912
4,2018-06-18,6,7176,INR,75378,543530
5,2017-11-20,59,500,USD,34323,158204


## 6. Customer Features (RFM) for Segmentation

Aggregates the cleaned `orders` table to one row per customer with the Recency / Frequency / Monetary features the README's K-Means segmentation step calls for, joined with demographic fields from `users`.

In [7]:
snapshot_date = orders["order_date"].max() + pd.Timedelta(days=1)

customer_features = (
    orders.groupby("user_id")
    .agg(
        total_orders=("order_date", "count"),
        total_spending=("sales_amount", "sum"),
        avg_order_value=("sales_amount", "mean"),
        total_items=("sales_qty", "sum"),
        first_order_date=("order_date", "min"),
        last_order_date=("order_date", "max"),
    )
    .reset_index()
)

customer_features["recency_days"] = (
    snapshot_date - customer_features["last_order_date"]
).dt.days

tenure_days = (
    customer_features["last_order_date"] - customer_features["first_order_date"]
).dt.days.clip(lower=1)
customer_features["order_frequency"] = customer_features["total_orders"] / tenure_days

customer_features = customer_features.merge(
    users[["user_id", "age", "gender", "occupation", "monthly_income", "family_size"]],
    on="user_id",
    how="left",
)

print(customer_features.shape)
customer_features.head()

(77223, 14)


,user_id,total_orders,total_spending,avg_order_value,total_items,first_order_date,last_order_date,recency_days,order_frequency,age,gender,occupation,monthly_income,family_size
0,1,2,1672,836.000000,5,2017-12-12,2018-11-14,591,0.005935,20,Female,Student,No Income,4
1,2,3,4061,1353.666667,7,2018-02-13,2018-08-03,694,0.017544,24,Female,Student,Below Rs.10000,3
2,3,1,21542,21542.000000,500,2019-05-15,2019-05-15,409,1.000000,22,Male,Student,Below Rs.10000,3
3,4,1,102,102.000000,1,2019-02-06,2019-02-06,507,1.000000,22,Female,Student,No Income,6
4,5,2,52047,26023.500000,50,2018-03-02,2018-07-25,703,0.013793,22,Male,Student,Below Rs.10000,4


## 7. Food Delivery Order History Data

- Rename columns to snake_case.
- Parse `Order Placed At` (e.g. `"11:38 PM, September 10 2024"`) into a real datetime.
- Convert `Distance` (`"3km"`, `"<1km"`) into a numeric `distance_km`.
- Parse the free-text `Items in order` (e.g. `"1 x Grilled Chicken Jamaican Tender, 1 x Grilled Chicken Peri Peri Tangdi"`) into a structured item list - this is the basket representation for the Apriori market basket analysis.
- Drop `Instructions` (96% missing free text, not used in any planned analysis).
- Leave `Rating`, `Review`, and cancellation/complaint fields as `NaN` where not applicable (e.g. an order that was never rated or cancelled) rather than dropping those rows.

In [8]:
order_history = pd.read_csv(
    RAW / "Food Delivery Order History Data/order_history_kaggle_data.csv"
)

order_history = order_history.rename(
    columns={
        "Restaurant ID": "restaurant_id",
        "Restaurant name": "restaurant_name",
        "Subzone": "subzone",
        "City": "city",
        "Order ID": "order_id",
        "Order Placed At": "order_placed_at",
        "Order Status": "order_status",
        "Delivery": "delivery_platform",
        "Distance": "distance",
        "Items in order": "items_in_order",
        "Instructions": "instructions",
        "Discount construct": "discount_construct",
        "Bill subtotal": "bill_subtotal",
        "Packaging charges": "packaging_charges",
        "Restaurant discount (Promo)": "restaurant_discount_promo",
        "Restaurant discount (Flat offs, Freebies & others)": "restaurant_discount_flat_other",
        "Gold discount": "gold_discount",
        "Brand pack discount": "brand_pack_discount",
        "Total": "total",
        "Rating": "rating",
        "Review": "review",
        "Cancellation / Rejection reason": "cancellation_reason",
        "Restaurant compensation (Cancellation)": "restaurant_compensation_cancellation",
        "Restaurant penalty (Rejection)": "restaurant_penalty_rejection",
        "KPT duration (minutes)": "kpt_duration_minutes",
        "Rider wait time (minutes)": "rider_wait_time_minutes",
        "Order Ready Marked": "order_ready_marked",
        "Customer complaint tag": "customer_complaint_tag",
        "Customer ID": "customer_id",
    }
)

order_history = order_history.drop(columns=["instructions"])

order_history["order_placed_at"] = pd.to_datetime(
    order_history["order_placed_at"], format="%I:%M %p, %B %d %Y"
)

order_history["distance_km"] = (
    order_history["distance"]
    .str.replace("<1km", "0.5", regex=False)
    .str.replace("km", "", regex=False)
    .astype(float)
)
order_history = order_history.drop(columns=["distance"])


def parse_items(text):
    parsed = []
    for part in str(text).split(","):
        match = re.match(r"\s*(\d+)\s*x\s*(.+)", part.strip())
        if match:
            parsed.append({"qty": int(match.group(1)), "item": match.group(2).strip()})
    return parsed


order_history["items_parsed"] = order_history["items_in_order"].apply(parse_items)
order_history["item_count"] = order_history["items_parsed"].apply(len)

order_history = order_history.drop_duplicates(subset="order_id").reset_index(drop=True)

print(order_history.shape)
print(order_history.isna().sum())
order_history.head()

(21321, 30)
restaurant_id                               0
restaurant_name                             0
subzone                                     0
city                                        0
order_id                                    0
order_placed_at                             0
order_status                                0
delivery_platform                           0
items_in_order                              0
discount_construct                       5498
bill_subtotal                               0
packaging_charges                           0
restaurant_discount_promo                   0
restaurant_discount_flat_other              0
gold_discount                               0
brand_pack_discount                         0
total                                       0
rating                                  18830
review                                  21025
cancellation_reason                     21135
restaurant_compensation_cancellation    21188
restaurant_penalty_rej

,restaurant_id,restaurant_name,subzone,city,order_id,order_placed_at,order_status,delivery_platform,items_in_order,discount_construct,bill_subtotal,packaging_charges,restaurant_discount_promo,restaurant_discount_flat_other,gold_discount,brand_pack_discount,total,rating,review,cancellation_reason,restaurant_compensation_cancellation,restaurant_penalty_rejection,kpt_duration_minutes,rider_wait_time_minutes,order_ready_marked,customer_complaint_tag,customer_id,distance_km,items_parsed,item_count
0,20320607,Swaad,Sector 4,Delhi NCR,6168884918,2024-09-10 23:38:00,Delivered,Zomato Delivery,"1 x Grilled Chicken Jamaican Tender, 1 x Grill...",40% off upto Rs.80,715.0,31.75,80.0,0.0,0.0,0.0,666.75,NaN,NaN,NaN,NaN,NaN,18.35,11.6,Correctly,NaN,5d6c2b96db963098bc69768bea504c8bf46106a8a5178e...,3.0,"[{'qty': 1, 'item': 'Grilled Chicken Jamaican ...",2
1,20320607,Swaad,Sector 4,Delhi NCR,6170707559,2024-09-10 23:34:00,Delivered,Zomato Delivery,"1 x Peri Peri Fries, 1 x Fried Chicken Angara ...",Flat Rs.175 off,1179.0,50.20,175.0,0.0,0.0,0.0,1054.20,NaN,NaN,NaN,NaN,NaN,16.95,3.6,Correctly,NaN,0781815deb4a10a574e9fee4fa0b86b074d4a0b36175d5...,2.0,"[{'qty': 1, 'item': 'Peri Peri Fries'}, {'qty'...",3
2,20320607,Swaad,Sector 4,Delhi NCR,6169375019,2024-09-10 15:52:00,Delivered,Zomato Delivery,1 x Bone in Peri Peri Grilled Chicken,40% off upto Rs.80,310.0,11.50,80.0,0.0,0.0,0.0,241.50,NaN,NaN,NaN,NaN,NaN,14.05,12.2,Correctly,NaN,f93362f5ce5382657482d164e368186bcec9c6225fd93d...,0.5,"[{'qty': 1, 'item': 'Bone in Peri Peri Grilled...",1
3,20320607,Swaad,Sector 4,Delhi NCR,6151677434,2024-09-10 15:45:00,Delivered,Zomato Delivery,"1 x Fried Chicken Ghostbuster Tender, 1 x Anga...",40% off upto Rs.80,620.0,27.00,80.0,0.0,0.0,0.0,567.00,4.0,NaN,NaN,NaN,NaN,19.00,3.3,Correctly,NaN,1ed226d1b8a5f7acee12fc1d6676558330a3b2b742af5d...,2.0,"[{'qty': 1, 'item': 'Fried Chicken Ghostbuster...",2
4,20320607,Swaad,Sector 4,Delhi NCR,6167540897,2024-09-10 15:04:00,Delivered,Zomato Delivery,"1 x Peri Peri Krispers, 1 x Fried Chicken Anga...",40% off upto Rs.80,584.0,25.20,80.0,0.0,0.0,0.0,529.20,NaN,NaN,NaN,NaN,NaN,15.97,1.0,Correctly,NaN,d21a2ac6ea06b31cc3288ab20c4ef2f292066c096f2c5f...,2.0,"[{'qty': 1, 'item': 'Peri Peri Krispers'}, {'q...",2


## 8. Zomato Delivery Operations Analytics Dataset

- Normalize column names; parse `Order_Date`, `Time_Orderd`, `Time_Order_picked`.
- Categorical fields (`weather_conditions`, `road_traffic_density`, `city`, `festival`) get an explicit `"Unknown"` category instead of `NaN` so rows aren't dropped just because one contextual field is missing.
- `multiple_deliveries` missing -> `0` (no concurrent delivery recorded).
- `delivery_person_age`/`delivery_person_ratings` (~4% missing each) are imputed with the column median rather than dropping rows, since every other field in those rows is valid.

In [9]:
delivery_ops = pd.read_csv(
    RAW / "Zomato Delivery Operations Analytics Dataset/Zomato Dataset.csv"
)

delivery_ops.columns = delivery_ops.columns.str.strip().str.lower()
delivery_ops = delivery_ops.rename(columns={"time_taken (min)": "time_taken_min"})

delivery_ops["order_date"] = pd.to_datetime(delivery_ops["order_date"], format="%d-%m-%Y")
delivery_ops["time_orderd"] = pd.to_datetime(
    delivery_ops["time_orderd"], format="%H:%M", errors="coerce"
).dt.time
delivery_ops["time_order_picked"] = pd.to_datetime(
    delivery_ops["time_order_picked"], format="%H:%M", errors="coerce"
).dt.time

for col in ["weather_conditions", "road_traffic_density", "city", "festival"]:
    delivery_ops[col] = delivery_ops[col].fillna("Unknown").astype("category")

delivery_ops["multiple_deliveries"] = delivery_ops["multiple_deliveries"].fillna(0).astype(int)
delivery_ops["delivery_person_age"] = delivery_ops["delivery_person_age"].fillna(
    delivery_ops["delivery_person_age"].median()
)
delivery_ops["delivery_person_ratings"] = delivery_ops["delivery_person_ratings"].fillna(
    delivery_ops["delivery_person_ratings"].median()
)

delivery_ops = delivery_ops.drop_duplicates(subset="id").reset_index(drop=True)

print(delivery_ops.shape)
print(delivery_ops.isna().sum())
delivery_ops.head()

(45584, 20)
id                                0
delivery_person_id                0
delivery_person_age               0
delivery_person_ratings           0
restaurant_latitude               0
restaurant_longitude              0
delivery_location_latitude        0
delivery_location_longitude       0
order_date                        0
time_orderd                    5799
time_order_picked              5007
weather_conditions                0
road_traffic_density              0
vehicle_condition                 0
type_of_order                     0
type_of_vehicle                   0
multiple_deliveries               0
festival                          0
city                              0
time_taken_min                    0
dtype: int64


,id,delivery_person_id,delivery_person_age,delivery_person_ratings,restaurant_latitude,restaurant_longitude,delivery_location_latitude,delivery_location_longitude,order_date,time_orderd,time_order_picked,weather_conditions,road_traffic_density,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city,time_taken_min
0,0xcdcd,DEHRES17DEL01,36.0,4.2,30.327968,78.046106,30.397968,78.116106,2022-02-12,21:55:00,22:10:00,Fog,Jam,2,Snack,motorcycle,3,No,Metropolitian,46
1,0xd987,KOCRES16DEL01,21.0,4.7,10.003064,76.307589,10.043064,76.347589,2022-02-13,14:55:00,15:05:00,Stormy,High,1,Meal,motorcycle,1,No,Metropolitian,23
2,0x2784,PUNERES13DEL03,23.0,4.7,18.562450,73.916619,18.652450,74.006619,2022-03-04,17:30:00,17:40:00,Sandstorms,Medium,1,Drinks,scooter,1,No,Metropolitian,21
3,0xc8b6,LUDHRES15DEL02,34.0,4.3,30.899584,75.809346,30.919584,75.829346,2022-02-13,09:20:00,09:30:00,Sandstorms,Low,0,Buffet,motorcycle,0,No,Metropolitian,20
4,0xdb64,KNPRES14DEL02,24.0,4.7,26.463504,80.372929,26.593504,80.502929,2022-02-14,19:50:00,20:05:00,Fog,Jam,1,Snack,scooter,1,No,Metropolitian,41


## 9. Save Processed Data

List-valued helper columns (`items_parsed`) don't round-trip through CSV, so they're dropped from the saved file - the parsed basket is easy to regenerate from `items_in_order` with `parse_items` above when building the market basket notebook.

In [10]:
outputs = {
    "users_clean.csv": users,
    "restaurants_clean.csv": restaurant,
    "food_clean.csv": food,
    "menu_clean.csv": menu,
    "orders_clean.csv": orders,
    "customer_features.csv": customer_features,
    "order_history_clean.csv": order_history.drop(columns=["items_parsed"]),
    "delivery_ops_clean.csv": delivery_ops,
}

for filename, df in outputs.items():
    df.to_csv(PROCESSED / filename, index=False)
    print(f"{filename:<26} {df.shape}")

users_clean.csv            (100000, 10)


restaurants_clean.csv      (148455, 8)


food_clean.csv             (371560, 3)


menu_clean.csv             (1178743, 5)


orders_clean.csv           (146979, 6)


customer_features.csv      (77223, 14)


order_history_clean.csv    (21321, 29)


delivery_ops_clean.csv     (45584, 20)


## Summary

| Output file | Purpose |
| --- | --- |
| `users_clean.csv` | Customer demographics |
| `restaurants_clean.csv` | Restaurant master data (rating, cost, cuisine) |
| `food_clean.csv` | Food item master data |
| `menu_clean.csv` | Restaurant-to-food price mapping |
| `orders_clean.csv` | Transaction-level sales facts |
| `customer_features.csv` | Per-customer RFM features for K-Means segmentation |
| `order_history_clean.csv` | Itemized orders for market basket analysis (Apriori) and order-level revenue/rating analysis |
| `delivery_ops_clean.csv` | Delivery-leg data for delivery performance analysis |

Next: `02_exploratory_analysis.ipynb`.